## create a spark session ##

In [77]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDD Learning") \
    .master("local[*]") \
    .getOrCreate()
print(spark)

## File location

In [78]:
import os

print(os.getcwd())

print("\nWORK FOLDERS:")
print(os.listdir("/home/jovyan/work"))

print("\nDATA FOLDERS:")
print(os.listdir("/home/jovyan/work/data"))

print("\nRAW FILES:")
print(os.listdir("/home/jovyan/work/data/raw"))

/home/jovyan

WORK FOLDERS:
['warehouse', 'data']

DATA FOLDERS:
['processed', 'raw']

RAW FILES:
['.ipynb_checkpoints', 'orders.csv']


In [62]:
import os

os.listdir("/home/jovyan/work/data/raw")

['.ipynb_checkpoints', 'orders.csv']

## Reads ordered file(csv) as RDD

In [79]:
orders_rdd = spark.sparkContext.textFile(
    "/home/jovyan/work/data/raw/orders.csv"
)

## Spark reads file 
Spark waits until an action is called.This is called Lazy Evaluation

In [64]:
orders_rdd.take(5)

['order_id,customer_id,product_id,quantity,price,order_timestamp',
 '1,101,P100,2,500,2026-01-01 10:00:00',
 '2,102,P101,1,200,2026-01-01 11:00:00',
 '3,101,P102,3,100,2026-01-02 09:00:00',
 '4,103,P103,2,700,2026-01-02 12:00:00']

## RDD operations are divided into transformations like map() and filter(), and actions like collect().

map() distributed computation.

In [ ]:
products_rdd = orders_rdd.map(
    lambda row: row.split(",")[2]   ##// Spark sends lambda function to EACH partition.
)
products_rdd.take(10)            ##//Each worker processes rows independently.

['product_id', 'P100', 'P101', 'P102', 'P103']

filter() Keeps only matching rows.

In [110]:
orders_rdd.filter
(lambda x: x.amount > 500)

<function __main__.<lambda>(x)>

In [85]:
high_qty_rdd = orders_rdd.filter(
    lambda row: int(row.split(",")[3]) > 2
)



flatmap() Flattens all values into single list.

In [ ]:
flat_rdd = orders_rdd.flatMap(
    lambda row: row.split(",")
)

flat_rdd.take(20)


['order_id',
 'customer_id',
 'product_id',
 'quantity',
 'price',
 'order_timestamp',
 '1',
 '101',
 'P100',
 '2',
 '500',
 '2026-01-01 10:00:00',
 '2',
 '102',
 'P101',
 '1',
 '200',
 '2026-01-01 11:00:00',
 '3',
 '101']

distinct()  Removes duplicates.

In [ ]:
us

['product_id', 'P102', 'P103', 'P100', 'P101']

repartition()

In [ ]:
new_rdd = orders_rdd.repartition(4) ## Change partition count

new_rdd.getNumPartitions()

4

mapreduce()

In [93]:
header = orders_rdd.first()

data_rdd = orders_rdd.filter(
    lambda row: row != header
)

qty_rdd = data_rdd.map(
    lambda row: int(row.split(",")[3])
)

total_qty = qty_rdd.reduce(
    lambda x,y: x+y
)

print(total_qty)
qty_rdd.take(10)

8


[2, 1, 3, 2]

## Lazy Evaluation: Most IMP SPARK Feature.
Transformations are lazy.

In [105]:
mapped_rdd = qty_rdd.map(
    lambda x: x * 2
)

filtered_rdd = mapped_rdd.filter(
    lambda x: x > 5              ##// NOTHING EXECUTES HERE Spark only builds DAG
)

total revenue

In [94]:
revenue_rdd = data_rdd.map(
    lambda row: int(row.split(",")[3]) * float(row.split(",")[4])
)

total_revenue = revenue_rdd.reduce(
    lambda x,y: x+y
)

print(total_revenue)

2900.0
